# AER reanalysis with the DISTILL core workflow

This notebook implements an inspectable staged workflow:

1. DISTILL hard cell QC and mitochondrial split
2. per-sample ASTRID annotation
3. manual cluster-prefix reannotation and explicit exclusions
4. per-sample kNN annotation of approved query cells
5. query-cell entropy QC and filtering at 0.3
6. BioLUCID batch-effect analysis checkpoint
7. mouse TIDYI only after BioLUCID review

The manual `sample + cluster prefix -> astrid_reannotation` rules are intentionally kept in this notebook. Low-quality, manual-Unannotated, and ASTRID-unassigned cells are excluded before kNN and cannot re-enter TIDYI. Expensive package calls remain guarded by explicit `RUN_*` switches.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path("/scratch/chenxi.sun/AER_DISTILL")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
RUNTIME_CACHE = PROJECT_ROOT / ".runtime_cache"
os.environ.setdefault("NUMBA_CACHE_DIR", str(RUNTIME_CACHE / "numba"))
os.environ.setdefault("MPLCONFIGDIR", str(RUNTIME_CACHE / "matplotlib"))
os.environ.setdefault("IPYTHONDIR", str(RUNTIME_CACHE / "ipython"))
Path(os.environ["NUMBA_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["IPYTHONDIR"]).mkdir(parents=True, exist_ok=True)

import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

from aer_distill_helpers import (
    apply_distill_qc,
    astrid_command_table,
    build_astrid_commands,
    ensure_marker_formula_copy,
    ensure_output_dirs,
    initialize_raw_counts,
    load_astrid_outputs,
    plot_celltype_percentages_by_sample,
    plot_sample_mean_umi,
    plot_tidyi_rates,
    prepare_astrid_inputs,
    prepare_mouse_tidyi,
    preview_biolucid_groups,
    restore_gene_symbols,
    run_astrid_commands,
    run_biolucid,
    run_growth_rate_decomposition_method3,
    run_tidyi_by_sample_celltype,
    save_qc_outputs,
    train_mouse_tidyi,
    transfer_labels_knn,
)

sc.settings.verbosity = 2
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 0
print(f"Python: {sys.executable}")
print(f"Scanpy: {sc.__version__}")

## 0. Paths and analysis parameters

The QC values below are the current DISTILL example values. Boundaries are inclusive: a cell with exactly 200 genes or 500 UMIs passes. Only cells with mitochondrial percentage strictly greater than 12.5 are withheld from ASTRID.

The ASTRID script/reference paths reproduce the working mouse invocation in `developmental_pattern.ipynb`. `cutoff_level=20` matches the DISTILL example configuration. The default mouse marker-formula CSV is copied once to the project `config` directory; edit that project-local file before running ASTRID validation.

In [ ]:
INPUT_H5AD = Path(
    "/scratch/chenxi.sun/AER2025_Juliane/scanpy_analysis/outputs/"
    "h5ad/aer_six_samples_raw_counts.h5ad"
)
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUT = ensure_output_dirs(OUTPUT_ROOT)

ASTRID_SCRIPT = Path(
    "/scratch/chenxi.sun/thesis_project/ASTRID_pkg/astrid/ASTRID_v0.01.py"
)
ASTRID_REFERENCE = Path(
    "/scratch/chenxi.sun/thesis_project/ASTRID_pkg/astrid/data/"
    "ASTRID_SingleR_Reference_Mukin_alltissues_20260204.Rds"
)
ASTRID_MARKER_FORMULA_SOURCE = Path(
    "/scratch/chenxi.sun/thesis_project/ASTRID_pkg/astrid/data/"
    "ExpectedCellTypesMarkersMouse.csv"
)
ASTRID_MARKER_FORMULA = ensure_marker_formula_copy(
    ASTRID_MARKER_FORMULA_SOURCE,
    PROJECT_ROOT / "config" / "ExpectedCellTypesMarkersMouse_AER.csv",
)

QC = {
    "min_genes_per_cell": 200,
    "max_genes_per_cell": 8000,
    "min_counts_per_cell": 500,
    "max_counts_per_cell": 50000,
    "max_pct_mito_for_astrid": 12.5,
}
ASTRID_CUTOFF_LEVEL = 20
BIOLUCID = {
    "batch_key": "sample",
    "cell_type_key": "cell_type",
    "min_cells": 20,
    "abundant_gene_threshold": 1,
    "min_abundant_genes": 100,
}
TRANSFER = {"n_neighbors": 10, "n_pca": 30, "random_state": RANDOM_STATE}
ENTROPY_THRESHOLD = 0.3
TIDYI = {
    "min_pct_genes_mapped": 70.0,
    "fail_pct_genes_mapped_lt": 30.0,
    "allow_low_mapping": False,
    "min_cells_exclusive": 50,  # earlier AER sample x cell-type rate rule
}

for required_path in [
    INPUT_H5AD,
    ASTRID_SCRIPT,
    ASTRID_REFERENCE,
    ASTRID_MARKER_FORMULA,
]:
    assert required_path.exists(), required_path

display(pd.Series({**QC, **BIOLUCID, **TRANSFER, **TIDYI}, name="value"))
print(f"Editable ASTRID marker formulas: {ASTRID_MARKER_FORMULA}")

## 1. Load the trusted raw-count reconstruction

This checks the expected AER metadata, preserves `X` as `layers['counts']`, and rejects matrices that do not look like non-negative integer counts. It does not modify the source file.

In [ ]:
adata_raw = initialize_raw_counts(sc.read_h5ad(INPUT_H5AD))

input_inventory = (
    adata_raw.obs.groupby(
        ["sample", "gsm", "genotype", "stage", "chemistry"],
        observed=True,
    )
    .size()
    .reset_index(name="n_cells")
)
display(input_inventory)
print(f"Input shape: {adata_raw.shape}; counts layer: {adata_raw.layers['counts'].shape}")
print(f"Author labels retained only for validation: {adata_raw.obs['author_cell_annotation'].notna().sum():,}")

## 2. DISTILL hard QC and ASTRID mitochondrial split

Four gene/count criteria permanently remove cells. No genes are filtered. High-mito cells remain in `full_post_qc` but are withheld from `for_astrid`; they are restored explicitly after ASTRID. No QC figure is generated.

In [ ]:
full_post_qc, for_astrid, qc_summary = apply_distill_qc(adata_raw, **QC)
display(qc_summary)

assert full_post_qc.n_vars == adata_raw.n_vars
assert for_astrid.n_vars == adata_raw.n_vars
save_qc_outputs(full_post_qc, for_astrid, qc_summary, OUT["qc"])
print(f"Saved hard-QC object: {OUT['qc'] / 'aer_full_post_qc.h5ad'}")
print(f"Saved ASTRID input population: {OUT['qc'] / 'aer_for_astrid.h5ad'}")

## 3. Prepare and inspect per-sample ASTRID commands

ASTRID receives raw counts, unique gene-symbol `var_names`, and the old author annotation only as `author_validation_label`. DISTILL runs ASTRID separately per sample, so this notebook does the same. The editable marker-formula CSV is passed explicitly to ASTRID validation. No `--plot` flag is passed.

In [ ]:
astrid_inputs = prepare_astrid_inputs(for_astrid, OUT["astrid"])
astrid_commands = build_astrid_commands(
    astrid_inputs,
    OUT["astrid"],
    astrid_script=ASTRID_SCRIPT,
    astrid_reference=ASTRID_REFERENCE,
    marker_formula=ASTRID_MARKER_FORMULA,
    cutoff_level=ASTRID_CUTOFF_LEVEL,
    python_executable=sys.executable,
)
astrid_commands_df = astrid_command_table(astrid_commands)
astrid_commands_df.to_csv(OUT["astrid"] / "astrid_commands.csv", index=False)
display(astrid_commands_df)

In [ ]:
# Expensive external step. Inspect astrid_commands_df first.
RUN_ASTRID = False

if RUN_ASTRID:
    astrid_run_summary = run_astrid_commands(astrid_commands, OUT["astrid"])
    astrid_run_summary.to_csv(OUT["astrid"] / "astrid_run_summary.csv", index=False)
    display(astrid_run_summary)
else:
    print("ASTRID not run. Set RUN_ASTRID=True after reviewing the commands.")

## 4. Load completed ASTRID outputs and normalize the six-sample CSV

This section resumes from the completed six per-sample ASTRID runs. It detects each sample's actual `clustering_level_*` result column, normalizes it to `final_clustering_level`, and creates the requested combined six-sample result table.

In [ ]:
if "full_post_qc" not in globals():
    full_post_qc = sc.read_h5ad(OUT["qc"] / "aer_full_post_qc.h5ad")
if "for_astrid" not in globals():
    for_astrid = sc.read_h5ad(OUT["qc"] / "aer_for_astrid.h5ad")

sample_order = for_astrid.obs["sample"].astype(str).drop_duplicates().tolist()
astrid_unreviewed, astrid_summary = load_astrid_outputs(OUT["astrid"], sample_order)
display(astrid_summary)

validation_tables = []
for sample in sample_order:
    result_path = OUT["astrid"] / sample / f"{sample}_ASTRID_results.csv"
    result = pd.read_csv(result_path)
    cluster_columns = [
        column
        for column in result.columns
        if column == "final_clustering_level" or column.startswith("clustering_level_")
    ]
    if len(cluster_columns) != 1:
        raise ValueError(f"{sample}: expected one clustering column, found {cluster_columns}")
    result = result.rename(columns={cluster_columns[0]: "final_clustering_level"})
    result.insert(0, "sample", sample)
    validation_tables.append(result)

combined_astrid_results = pd.concat(validation_tables, ignore_index=True)
COMBINED_ASTRID_RESULTS_CSV = OUT["astrid"] / "astrid_all_samples_results.csv"
combined_astrid_results.to_csv(COMBINED_ASTRID_RESULTS_CSV, index=False)
print(f"Saved combined six-sample ASTRID results: {COMBINED_ASTRID_RESULTS_CSV}")

## 5. Manual ASTRID cluster reannotation

The mapping below is the authoritative manual review. Cluster paths are compared after removing underscores, and the longest matching prefix wins. For example, `0412` overrides the broader `041` rule. The mapping is deliberately visible here rather than hidden in a helper module.

In [ ]:
ASTRID_REANNOTATION_PREFIXES = {
    "GSM7876383_WT_E95": {
        "00": "Mesenchyme", "01": "Mesenchyme", "02": "Mesenchyme",
        "03": "Muscle", "10": "Low quality", "11": "Low quality",
        "12": "Erythrocytes", "200": "AER", "202": "AER",
        "203": "AER", "22": "AER", "201": "DV", "204": "DV",
        "21": "DV", "23": "DV", "3": "Vascular cells",
    },
    "GSM7876384_Dac_E95": {
        "00": "Mesenchyme", "01": "Mesenchyme", "02": "Mesenchyme",
        "034": "Mesenchyme", "030": "Muscle", "031": "Muscle",
        "032": "Muscle", "033": "Muscle", "10": "AER", "11": "DV",
        "14": "DV", "15": "DV", "12": "Erythrocytes",
        "13": "Mesenchyme", "2": "Vascular cells",
    },
    "GSM7876385_WT_E105": {
        "00": "Mesenchyme", "01": "Mesenchyme", "021": "Mesenchyme",
        "020": "Muscle", "022": "Muscle", "023": "Vascular cells",
        "10": "Vascular cells", "12": "Vascular cells", "13": "Vascular cells",
        "14": "Vascular cells", "030": "DV", "031": "DV", "033": "DV",
        "034": "DV", "032": "AER", "11": "Immune", "15": "Immune",
        "20": "Erythrocytes", "21": "Erythrocytes", "22": "Erythrocytes",
    },
    "GSM7876386_Dac_E105": {
        "0": "Mesenchyme", "10": "Mesenchyme", "11": "Mesenchyme",
        "12": "Muscle", "20": "DV", "26": "DV", "21": "Vascular cells",
        "22": "Erythrocytes", "24": "AER", "25": "Immune",
        "23": "Unannotated",
    },
    "GSM7876387_WT_E115": {
        "00": "Mesenchyme", "01": "Mesenchyme", "020": "DV", "021": "DV",
        "023": "DV", "022": "AER", "03": "Muscle", "04": "Vascular cells",
        "05": "Immune", "10": "Low quality", "11": "Low quality",
        "22": "Low quality", "20": "Erythrocytes", "21": "Erythrocytes",
        "23": "Erythrocytes",
    },
    "GSM7876388_Dac_E115": {
        "00": "Mesenchyme", "01": "Mesenchyme", "02": "Mesenchyme",
        "03": "Mesenchyme", "040": "Muscle", "041": "Muscle",
        "042": "Muscle", "043": "Muscle", "045": "Muscle",
        "044": "Vascular cells", "046": "Immune", "10": "Low quality",
        "11": "Low quality", "0412": "Low quality", "12": "Erythrocytes",
        "13": "Low quality",
    },
}


def manual_astrid_label(sample, cluster):
    cluster = str(cluster)
    if cluster == "unassigned":
        return "Unassigned"
    compact = cluster.replace("_", "")
    matches = [
        (prefix, label)
        for prefix, label in ASTRID_REANNOTATION_PREFIXES[str(sample)].items()
        if compact.startswith(prefix)
    ]
    if not matches:
        return pd.NA
    longest = max(len(prefix) for prefix, _ in matches)
    labels = {label for prefix, label in matches if len(prefix) == longest}
    if len(labels) != 1:
        raise ValueError(f"Ambiguous mapping for {sample} cluster {cluster}: {matches}")
    return labels.pop()


astrid_unreviewed.obs["astrid_reannotation"] = pd.array(
    [
        manual_astrid_label(sample, cluster)
        for sample, cluster in zip(
            astrid_unreviewed.obs["sample"],
            astrid_unreviewed.obs["final_clustering_level"],
        )
    ],
    dtype="string",
)

cluster_mapping_audit = (
    astrid_unreviewed.obs
    .groupby(
        ["sample", "final_clustering_level", "astrid_reannotation"],
        observed=True,
        dropna=False,
    )
    .size()
    .reset_index(name="n_cells")
)
missing_mapping = cluster_mapping_audit["astrid_reannotation"].isna()
display(cluster_mapping_audit.loc[missing_mapping])
assert not missing_mapping.any(), "At least one ASTRID cluster is missing a manual mapping"

combined_astrid_results["astrid_reannotation"] = pd.array(
    [
        manual_astrid_label(sample, cluster)
        for sample, cluster in zip(
            combined_astrid_results["sample"],
            combined_astrid_results["final_clustering_level"],
        )
    ],
    dtype="string",
)
combined_astrid_results.to_csv(COMBINED_ASTRID_RESULTS_CSV, index=False)
cluster_mapping_audit.to_csv(OUT["astrid"] / "astrid_reannotation_cluster_audit.csv", index=False)
astrid_unreviewed.write_h5ad(
    OUT["astrid"] / "aer_low_mito_astrid_reannotated_before_exclusion.h5ad",
    compression="gzip",
)
display(
    pd.crosstab(
        astrid_unreviewed.obs["sample"],
        astrid_unreviewed.obs["astrid_reannotation"],
    )
)

## 6. Explicit exclusions and reference/query definition

Deletion is applied before reference/query selection. `Low quality`, manual `Unannotated`, and ASTRID `unassigned` barcodes are removed. Query cells are the remaining original author-unannotated cells plus all `qc_high_mito` cells. Reference cells are non-query cells with a valid `astrid_reannotation`.

In [ ]:
exclusion_reason = pd.Series(
    pd.NA, index=astrid_unreviewed.obs_names, dtype="string", name="exclusion_reason"
)
exclusion_reason.loc[
    astrid_unreviewed.obs["astrid_reannotation"].eq("Low quality")
] = "Low quality"
exclusion_reason.loc[
    astrid_unreviewed.obs["astrid_reannotation"].eq("Unannotated")
] = "Manual Unannotated"
exclusion_reason.loc[
    astrid_unreviewed.obs["final_clustering_level"].astype(str).eq("unassigned")
] = "ASTRID unassigned"

astrid_unreviewed.obs["exclusion_reason"] = exclusion_reason
excluded_low_mito = astrid_unreviewed.obs.loc[exclusion_reason.notna()].copy()
astrid_retained = astrid_unreviewed[exclusion_reason.isna()].copy()

full_keep = ~full_post_qc.obs_names.isin(excluded_low_mito.index)
pre_knn = full_post_qc[full_keep].copy()
pre_knn.X = pre_knn.layers["counts"].copy()
pre_knn.obs["final_clustering_level"] = pd.Series(
    pd.NA, index=pre_knn.obs_names, dtype="string"
)
pre_knn.obs["astrid_reannotation"] = pd.Series(
    pd.NA, index=pre_knn.obs_names, dtype="string"
)
for column in ["final_clustering_level", "astrid_reannotation"]:
    pre_knn.obs.loc[astrid_retained.obs_names, column] = (
        astrid_retained.obs[column].astype("string").to_numpy()
    )

author_annotated = pre_knn.obs["is_author_annotated"].astype("boolean").fillna(False)
query_high_mito = pre_knn.obs["qc_high_mito"].astype(bool)
query_author_unannotated = ~author_annotated
query_mask = query_high_mito | query_author_unannotated
reference_mask = (~query_mask) & pre_knn.obs["astrid_reannotation"].notna()

assert not (query_mask & reference_mask).any()
assert (query_mask | reference_mask).all(), "Some retained cells are neither reference nor query"
assert not pre_knn.obs_names.isin(excluded_low_mito.index).any()

pre_knn.obs["knn_role"] = pd.Series(pd.NA, index=pre_knn.obs_names, dtype="string")
pre_knn.obs.loc[reference_mask, "knn_role"] = "reference"
pre_knn.obs.loc[query_mask, "knn_role"] = "query"
pre_knn.obs["knn_query_author_unannotated"] = query_author_unannotated.to_numpy()
pre_knn.obs["knn_query_high_mito"] = query_high_mito.to_numpy()
pre_knn.obs["cell_type"] = pd.Series(pd.NA, index=pre_knn.obs_names, dtype="string")
pre_knn.obs.loc[reference_mask, "cell_type"] = (
    pre_knn.obs.loc[reference_mask, "astrid_reannotation"].astype("string")
)
pre_knn.obs["cell_type_source"] = pd.Series(pd.NA, index=pre_knn.obs_names, dtype="string")
pre_knn.obs.loc[reference_mask, "cell_type_source"] = "astrid_reannotation_reference"
pre_knn.obs.loc[query_mask, "cell_type_source"] = "knn_query_pending"
pre_knn.obs["cell_type_transfer_confidence"] = np.nan
pre_knn.obs["cell_type_transfer_entropy"] = np.nan

role_audit = (
    pre_knn.obs.groupby(["sample", "knn_role"], observed=True)
    .size()
    .unstack(fill_value=0)
)
reference_celltypes = pd.crosstab(
    pre_knn.obs.loc[reference_mask, "sample"],
    pre_knn.obs.loc[reference_mask, "cell_type"],
)
exclusion_counts = (
    excluded_low_mito.groupby(["sample", "exclusion_reason"], observed=True)
    .size()
    .unstack(fill_value=0)
)
display(exclusion_counts)
display(role_audit)
display(reference_celltypes)
exclusion_counts.to_csv(OUT["astrid"] / "excluded_low_mito_counts.csv")
role_audit.to_csv(OUT["restore"] / "reference_query_counts.csv")
reference_celltypes.to_csv(OUT["restore"] / "reference_celltype_counts.csv")

## 7. Per-sample kNN annotation and entropy QC

Each sample is processed independently with median-depth normalization, log1p, PCA, and Euclidean kNN. Checkpoints contain only per-query predictions and can be safely resumed when the exact reference/query signatures match. Entropy filtering applies only to query cells: `< 0.3` is retained and `>= 0.3` is removed.

In [ ]:
RUN_KNN = False
KNN_CHECKPOINT_DIR = OUT["restore"] / "per_sample_knn"
KNN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
POST_ENTROPY_H5AD = OUT["restore"] / "aer_post_entropy.h5ad"

if RUN_KNN:
    import hashlib

    transfer_rows = []
    for sample in sample_order:
        in_sample = pre_knn.obs["sample"].astype(str).eq(sample)
        sample_ref = pre_knn.obs_names[in_sample & reference_mask]
        sample_query = pre_knn.obs_names[in_sample & query_mask]
        if len(sample_ref) < 2:
            raise ValueError(f"{sample}: fewer than two reference cells")

        ref_signature = hashlib.sha256(
            "\n".join(
                f"{name}	{pre_knn.obs.at[name, 'cell_type']}" for name in sample_ref
            ).encode()
        ).hexdigest()
        query_signature = hashlib.sha256("\n".join(sample_query).encode()).hexdigest()
        checkpoint_csv = KNN_CHECKPOINT_DIR / f"{sample}_predictions.csv"
        checkpoint_json = KNN_CHECKPOINT_DIR / f"{sample}_metadata.json"
        use_checkpoint = False
        if checkpoint_csv.exists() and checkpoint_json.exists():
            metadata = json.loads(checkpoint_json.read_text())
            use_checkpoint = (
                metadata.get("ref_signature") == ref_signature
                and metadata.get("query_signature") == query_signature
                and metadata.get("transfer") == TRANSFER
            )

        if use_checkpoint:
            predictions = pd.read_csv(checkpoint_csv, index_col="barcode")
            predictions = predictions.loc[sample_query]
            print(f"{sample}: loaded matching kNN checkpoint ({len(sample_query)} query cells)")
        else:
            print(
                f"{sample}: running kNN with {len(sample_ref)} reference and "
                f"{len(sample_query)} query cells",
                flush=True,
            )
            ref_matrix = pre_knn[sample_ref].layers["counts"]
            query_matrix = pre_knn[sample_query].layers["counts"]
            predicted, entropy, confidence = transfer_labels_knn(
                ref_matrix,
                query_matrix,
                pre_knn.obs.loc[sample_ref, "cell_type"].astype(str).to_numpy(),
                **TRANSFER,
            )
            predictions = pd.DataFrame(
                {
                    "cell_type": predicted,
                    "cell_type_transfer_entropy": entropy,
                    "cell_type_transfer_confidence": confidence,
                },
                index=pd.Index(sample_query, name="barcode"),
            )
            predictions.to_csv(checkpoint_csv)
            checkpoint_json.write_text(
                json.dumps(
                    {
                        "sample": sample,
                        "ref_signature": ref_signature,
                        "query_signature": query_signature,
                        "transfer": TRANSFER,
                        "n_reference": len(sample_ref),
                        "n_query": len(sample_query),
                    },
                    indent=2,
                )
                + "\n"
            )

        pre_knn.obs.loc[sample_query, "cell_type"] = predictions["cell_type"].astype("string")
        pre_knn.obs.loc[sample_query, "cell_type_transfer_entropy"] = predictions[
            "cell_type_transfer_entropy"
        ].to_numpy()
        pre_knn.obs.loc[sample_query, "cell_type_transfer_confidence"] = predictions[
            "cell_type_transfer_confidence"
        ].to_numpy()
        pre_knn.obs.loc[sample_query, "cell_type_source"] = "knn_transferred"
        transfer_rows.append(
            {
                "sample": sample,
                "n_reference": len(sample_ref),
                "n_query": len(sample_query),
                "n_common_genes": pre_knn.n_vars,
                "mean_confidence": predictions["cell_type_transfer_confidence"].mean(),
                "median_confidence": predictions["cell_type_transfer_confidence"].median(),
                "mean_entropy": predictions["cell_type_transfer_entropy"].mean(),
                "median_entropy": predictions["cell_type_transfer_entropy"].median(),
            }
        )

    transfer_summary = pd.DataFrame(transfer_rows)
    query_entropy = pre_knn.obs.loc[query_mask, "cell_type_transfer_entropy"]
    assert query_entropy.notna().all(), "At least one query cell has no entropy value"
    pre_knn.obs["entropy_class"] = pd.Series(pd.NA, index=pre_knn.obs_names, dtype="string")
    pre_knn.obs.loc[query_mask & (pre_knn.obs["cell_type_transfer_entropy"] < ENTROPY_THRESHOLD), "entropy_class"] = "low_entropy"
    pre_knn.obs.loc[query_mask & (pre_knn.obs["cell_type_transfer_entropy"] >= ENTROPY_THRESHOLD), "entropy_class"] = "high_entropy"

    query_obs = pre_knn.obs.loc[query_mask].copy()
    entropy_qc_by_sample = (
        query_obs.groupby(["sample", "entropy_class"], observed=True)
        .agg(
            n_cells=("cell_type", "size"),
            umi_median=("total_counts", "median"),
            umi_q25=("total_counts", lambda values: values.quantile(0.25)),
            umi_q75=("total_counts", lambda values: values.quantile(0.75)),
            genes_median=("n_genes_by_counts", "median"),
            genes_q25=("n_genes_by_counts", lambda values: values.quantile(0.25)),
            genes_q75=("n_genes_by_counts", lambda values: values.quantile(0.75)),
            mt_pct_median=("pct_counts_mt", "median"),
            mt_pct_q25=("pct_counts_mt", lambda values: values.quantile(0.25)),
            mt_pct_q75=("pct_counts_mt", lambda values: values.quantile(0.75)),
            confidence_median=("cell_type_transfer_confidence", "median"),
            entropy_median=("cell_type_transfer_entropy", "median"),
        )
        .reset_index()
    )
    entropy_qc_by_celltype = (
        query_obs.groupby(["sample", "entropy_class", "cell_type"], observed=True)
        .agg(
            n_cells=("cell_type", "size"),
            umi_median=("total_counts", "median"),
            genes_median=("n_genes_by_counts", "median"),
            mt_pct_median=("pct_counts_mt", "median"),
            confidence_median=("cell_type_transfer_confidence", "median"),
            entropy_median=("cell_type_transfer_entropy", "median"),
        )
        .reset_index()
    )

    entropy_keep = (~query_mask) | (
        query_mask & (pre_knn.obs["cell_type_transfer_entropy"] < ENTROPY_THRESHOLD)
    )
    entropy_keep_array = entropy_keep.to_numpy(dtype=bool)
    high_entropy_mask = (query_mask & ~entropy_keep).to_numpy(dtype=bool)
    post_entropy = pre_knn[entropy_keep_array].copy()
    high_entropy_obs = pre_knn.obs.loc[high_entropy_mask].copy()

    assert post_entropy.obs["cell_type"].notna().all()
    assert not post_entropy.obs_names.isin(excluded_low_mito.index).any()
    assert not post_entropy.obs_names.isin(high_entropy_obs.index).any()
    assert (
        post_entropy.obs.loc[
            post_entropy.obs["knn_role"].eq("query"),
            "cell_type_transfer_entropy",
        ] < ENTROPY_THRESHOLD
    ).all()

    transfer_summary.to_csv(OUT["restore"] / "knn_transfer_summary.csv", index=False)
    entropy_qc_by_sample.to_csv(OUT["restore"] / "entropy_qc_by_sample.csv", index=False)
    entropy_qc_by_celltype.to_csv(
        OUT["restore"] / "entropy_qc_by_sample_celltype.csv", index=False
    )
    high_entropy_obs.to_csv(OUT["restore"] / "excluded_high_entropy_cells.csv")
    post_entropy.write_h5ad(POST_ENTROPY_H5AD, compression="gzip")
else:
    if not POST_ENTROPY_H5AD.exists():
        raise RuntimeError("Set RUN_KNN=True; no completed post-entropy object exists")
    post_entropy = sc.read_h5ad(POST_ENTROPY_H5AD)
    transfer_summary = pd.read_csv(OUT["restore"] / "knn_transfer_summary.csv")
    entropy_qc_by_sample = pd.read_csv(OUT["restore"] / "entropy_qc_by_sample.csv")
    entropy_qc_by_celltype = pd.read_csv(
        OUT["restore"] / "entropy_qc_by_sample_celltype.csv"
    )

display(transfer_summary)
display(entropy_qc_by_sample)
display(entropy_qc_by_celltype)

## 8. Post-filter composition and BioLUCID checkpoint

BioLUCID receives the post-entropy object. It is diagnostic: it does not automatically delete cells, cell types, or samples. Review its global and per-sample output before enabling TIDYI.

In [ ]:
prohibited = {"Low quality", "Unannotated", "Unassigned"}
observed_labels = set(post_entropy.obs["cell_type"].astype(str))
assert not (observed_labels & prohibited), observed_labels & prohibited
assert post_entropy.obs["cell_type"].notna().all()

post_filter_counts = pd.crosstab(post_entropy.obs["sample"], post_entropy.obs["cell_type"])
biolucid_counts, biolucid_valid_celltypes = preview_biolucid_groups(
    post_entropy,
    batch_key=BIOLUCID["batch_key"],
    cell_type_key=BIOLUCID["cell_type_key"],
    min_cells=BIOLUCID["min_cells"],
)
post_filter_counts.to_csv(OUT["restore"] / "post_entropy_celltype_by_sample.csv")
biolucid_counts.to_csv(OUT["biolucid"] / "celltype_by_sample_input_counts.csv")
display(post_filter_counts)
print(f"BioLUCID-eligible cell types at n >= {BIOLUCID['min_cells']} in every sample:")
display(pd.Series(biolucid_valid_celltypes, name="retained_cell_type"))

In [ ]:
RUN_BIOLUCID = False

if RUN_BIOLUCID:
    biolucid_analyzer, biolucid_global, biolucid_per_sample, biolucid_figure = run_biolucid(
        post_entropy, OUT["biolucid"], **BIOLUCID
    )
    display(biolucid_global)
    display(biolucid_per_sample)
    print("BioLUCID complete. Stop here for manual batch-effect review before TIDYI.")
else:
    print("BioLUCID not run. Set RUN_BIOLUCID=True after reviewing its input table.")

## 9. Prepare mouse Ensembl genes for TIDYI — run only after BioLUCID approval

TIDYI reads the already filtered `post_entropy` object. It does not reload `full_post_qc`, so excluded low-quality, manual-Unannotated, ASTRID-unassigned, and high-entropy barcodes cannot return.

In [ ]:
RUN_TIDYI_PREPARATION = False

if RUN_TIDYI_PREPARATION:
    tidyi_input, tidyi_gene_report = prepare_mouse_tidyi(
        post_entropy,
        min_pct_genes_mapped=TIDYI["min_pct_genes_mapped"],
        fail_pct_genes_mapped_lt=TIDYI["fail_pct_genes_mapped_lt"],
        allow_low_mapping=TIDYI["allow_low_mapping"],
    )
    display(tidyi_gene_report)
    tidyi_gene_report.to_csv(OUT["tidyi"] / "gene_mapping_report.csv", index=False)
    tidyi_input.var_names.to_series().to_csv(
        OUT["tidyi"] / "tidyi_genes_used.txt", index=False, header=False
    )
    print("TIDYI input prepared in memory; only mapping reports are persisted.")
else:
    print("Stopped before TIDYI preparation pending BioLUCID review.")

## 10. Train and apply TIDYI

Every retained cell receives per-cell division/death scores. The `n_cells > 50` rule only controls sample × cell-type rate-summary rows; it does not remove cells.

In [ ]:
RUN_TIDYI_TRAINING = False

if RUN_TIDYI_TRAINING:
    assert "tidyi_input" in globals(), "Run TIDYI preparation first"
    tidyi_model = train_mouse_tidyi(tidyi_input.var_names, random_state=RANDOM_STATE)
else:
    print("TIDYI training not run.")

In [ ]:
RUN_TIDYI_APPLICATION = False

if RUN_TIDYI_APPLICATION:
    assert "tidyi_model" in globals(), "Run TIDYI training first"
    tidyi_scored, tidyi_group_counts, tidyi_rates = run_tidyi_by_sample_celltype(
        tidyi_input,
        tidyi_model,
        sample_key="sample",
        cell_type_key="cell_type",
        min_cells_exclusive=TIDYI["min_cells_exclusive"],
    )
    tidyi_group_counts.to_csv(OUT["tidyi"] / "sample_celltype_group_counts.csv", index=False)
    tidyi_rates.to_csv(OUT["tidyi"] / "sample_celltype_rates.csv", index=False)
    display(tidyi_group_counts)
    display(tidyi_rates)
    tidyi_rate_figure = plot_tidyi_rates(
        tidyi_rates, OUT["tidyi"] / "tidyi_rates.png"
    )
    plt.show()
else:
    print("TIDYI application not run.")

## 11. Final export and reconciliation

The final audit requires exact equality with the post-entropy retained barcode set, complete labels and scores, and absence of every explicitly excluded barcode.

In [ ]:
RUN_FINAL_EXPORT = False

if RUN_FINAL_EXPORT:
    assert "tidyi_scored" in globals(), "Run TIDYI application first"
    final_adata = restore_gene_symbols(tidyi_scored)
    assert final_adata.obs_names.equals(post_entropy.obs_names)
    assert final_adata.obs["cell_type"].notna().all()
    assert final_adata.obs[["division_score", "death_score"]].notna().all().all()
    assert not final_adata.obs_names.isin(excluded_low_mito.index).any()

    final_counts = pd.crosstab(final_adata.obs["sample"], final_adata.obs["cell_type"])
    final_counts.to_csv(OUT["final"] / "final_celltype_by_sample.csv")
    final_path = OUT["final"] / "aer_distill_final.h5ad"
    final_adata.write_h5ad(final_path, compression="gzip")

    excluded_high_entropy = pd.read_csv(
        OUT["restore"] / "excluded_high_entropy_cells.csv", index_col=0
    )
    roundtrip = ad.read_h5ad(final_path, backed="r")
    prohibited_labels = {"Low quality", "Unannotated", "Unassigned"}
    score_values = roundtrip.obs[["division_score", "death_score"]].to_numpy(dtype=float)
    assert roundtrip.obs_names.equals(post_entropy.obs_names)
    assert roundtrip.obs["sample"].astype(str).nunique() == len(sample_order)
    assert roundtrip.obs["cell_type"].notna().all()
    assert not (set(roundtrip.obs["cell_type"].astype(str)) & prohibited_labels)
    assert np.isfinite(score_values).all()
    assert not roundtrip.obs_names.isin(excluded_low_mito.index).any()
    assert not roundtrip.obs_names.isin(excluded_high_entropy.index).any()
    assert roundtrip.var_names.is_unique
    assert roundtrip.var.index.name is None

    audit_summary = {
        "n_cells": int(roundtrip.n_obs),
        "n_genes": int(roundtrip.n_vars),
        "n_samples": int(roundtrip.obs["sample"].astype(str).nunique()),
        "n_cell_types": int(roundtrip.obs["cell_type"].astype(str).nunique()),
        "missing_cell_type": int(roundtrip.obs["cell_type"].isna().sum()),
        "missing_tidyi_scores": int(np.isnan(score_values).sum()),
        "excluded_low_mito_reentered": int(
            roundtrip.obs_names.isin(excluded_low_mito.index).sum()
        ),
        "excluded_high_entropy_reentered": int(
            roundtrip.obs_names.isin(excluded_high_entropy.index).sum()
        ),
        "barcode_set_and_order_match_post_entropy": bool(
            roundtrip.obs_names.equals(post_entropy.obs_names)
        ),
        "var_index_name_is_none": roundtrip.var.index.name is None,
    }
    roundtrip.file.close()
    (OUT["final"] / "final_audit_summary.json").write_text(
        json.dumps(audit_summary, indent=2) + "\n"
    )
    display(final_counts)
    display(pd.Series(audit_summary, name="value"))
    print("FINAL_AUDIT_OK")
else:
    print("Final export not run.")

## 12. Cell-type percentages by sample

Reproduce the legacy `TIDYI_cellrangerAER.ipynb` grouped bar chart using the final DISTILL-retained cells. Percentages use retained cells within each sample as the denominator. The figure keeps the original sample order, colors, grouped-bar dimensions, log10 y-axis, legend, rotation, and 300-dpi export. True zero-percent groups are absent because zero cannot be represented on a logarithmic axis.

In [ ]:
DISTILL_FINAL_H5AD = OUT["final"] / "aer_distill_final.h5ad"
CELLTYPE_PERCENTAGE_OUTPUT_DIR = (
    OUT["final"] / "celltype_percentage_barplots"
)

celltype_percentage_table, celltype_percentage_figure = (
    plot_celltype_percentages_by_sample(
        input_path=DISTILL_FINAL_H5AD,
        output_dir=CELLTYPE_PERCENTAGE_OUTPUT_DIR,
    )
)
display(celltype_percentage_table)
plt.show()

## 13. Growth-rate decomposition — Method 3

Apply the noise-weighted Method 3 to the final DISTILL TIDYI sample × cell-type rates. Method 3 uses `|Δdeath| / (|Δdeath| + k × |Δproliferation|)`. Because AER has one sample per genotype × stage and therefore cannot supply replicate CVs, the proliferation/death CVs are taken from the Mukin mouse reference. The global mean rates are instead calculated as unweighted means across the 34 available AER sample × cell-type rate rows: `k = (CV_death × AER mean death rate) / (CV_proliferation × AER mean proliferation rate)`, which gives approximately `0.267188`. The former Mukin-global-mean `k` is retained in the audit table as a sensitivity reference. The seven contrasts below are all defined as condition A minus condition B. The filtered heatmap applies the original regulation-magnitude cutoff `|Δproliferation| + |Δdeath| ≥ 2 × log(2) / (24 × 3)`. The accompanying 5-bin histogram follows the original before/after-filter layout, fixed 0–1 bins, percentage normalization, colors, and after-filter rug marks. Gray heatmap cells lack a TIDYI rate in at least one condition because the existing rate table only reports groups with more than 50 cells.


In [ ]:
METHOD3_TYPICAL_CV_PATH = Path(
    "/scratch/chenxi.sun/TIDYI_statistics/results/aging_paired_ttest/"
    "effect_size_typical_cv/stats_notebook_aging_typical_cv_values.csv"
)
METHOD3_REFERENCE_CV_DETAIL_PATH = Path(
    "/scratch/chenxi.sun/TIDYI_statistics/results/ref_cv/"
    "stats_notebook_ref_cv_separate_by_dataset_celltype.csv"
)
REGULATION_MAGNITUDE_THRESHOLD = 2 * np.log(2) / (24 * 3)
AER_METHOD3_CONTRASTS = [
    {
        "contrast": "WT E10.5 - E9.5",
        "genotype_a": "WT", "stage_a": "E10.5",
        "genotype_b": "WT", "stage_b": "E9.5",
    },
    {
        "contrast": "WT E11.5 - E10.5",
        "genotype_a": "WT", "stage_a": "E11.5",
        "genotype_b": "WT", "stage_b": "E10.5",
    },
    {
        "contrast": "Dac E10.5 - E9.5",
        "genotype_a": "Dac", "stage_a": "E10.5",
        "genotype_b": "Dac", "stage_b": "E9.5",
    },
    {
        "contrast": "Dac E11.5 - E10.5",
        "genotype_a": "Dac", "stage_a": "E11.5",
        "genotype_b": "Dac", "stage_b": "E10.5",
    },
    {
        "contrast": "E9.5 Dac - WT",
        "genotype_a": "Dac", "stage_a": "E9.5",
        "genotype_b": "WT", "stage_b": "E9.5",
    },
    {
        "contrast": "E10.5 Dac - WT",
        "genotype_a": "Dac", "stage_a": "E10.5",
        "genotype_b": "WT", "stage_b": "E10.5",
    },
    {
        "contrast": "E11.5 Dac - WT",
        "genotype_a": "Dac", "stage_a": "E11.5",
        "genotype_b": "WT", "stage_b": "E11.5",
    },
]

(
    method3_results,
    method3_unfiltered_figure,
    method3_filtered_figure,
    method3_histogram_figure,
) = (
    run_growth_rate_decomposition_method3(
        rates_path=OUT["tidyi"] / "sample_celltype_rates.csv",
        typical_cv_path=METHOD3_TYPICAL_CV_PATH,
        reference_cv_detail_path=METHOD3_REFERENCE_CV_DETAIL_PATH,
        output_dir=OUT["growth_decomposition"],
        contrasts=AER_METHOD3_CONTRASTS,
        cv_dataset="Mukin",
        regulation_magnitude_threshold=REGULATION_MAGNITUDE_THRESHOLD,
    )
)
method3_summary = (
    method3_results.groupby("contrast", sort=False)
    .agg(
        fractions_available=("available_method3_fraction", "sum"),
        fractions_passing_filter=("passes_magnitude_filter", "sum"),
    )
)
display(
    pd.Series(
        {
            "method3_k": method3_results["method3_k"].iloc[0],
            "method3_k_mukin_reference_sensitivity": method3_results[
                "method3_k_mukin_reference_sensitivity"
            ].iloc[0],
            "proliferation_global_mean_used": method3_results[
                "proliferation_global_mean_used"
            ].iloc[0],
            "death_global_mean_used": method3_results[
                "death_global_mean_used"
            ].iloc[0],
            "global_mean_rate_source": method3_results[
                "global_mean_rate_source"
            ].iloc[0],
            "n_aer_rate_rows_for_global_mean": method3_results[
                "n_aer_rate_rows_for_global_mean"
            ].iloc[0],
            "regulation_threshold_h^-1": REGULATION_MAGNITUDE_THRESHOLD,
            "cv_dataset": method3_results["cv_dataset"].iloc[0],
        },
        name="value",
    )
)
display(method3_summary)
display(method3_results)
plt.show()

## 14. Mean UMI counts by sample

Plot one bar per sample and one figure per dataset using the raw-count `total_counts` field. AER uses the final post-entropy DISTILL cells. Mukin and TMS use the tissue-wise kNN H5AD objects restricted to the same sample lists used by the previous TIDYI analysis. The count field was checked against the corresponding raw-count layer before plotting. Each panel's red dashed line is the cell-weighted mean UMI count across all retained cells in that dataset.

In [ ]:
AER_FINAL_PATH = OUT["final"] / "aer_distill_final.h5ad"
MUKIN_KNN_PATH = Path(
    "/scratch/chenxi.sun/thesis_project/mukin_qcdata/"
    "mukin_data_tissuewise_knn.h5ad"
)
TMS_KNN_PATH = Path(
    "/scratch/chenxi.sun/thesis_project/tms_qcdata/"
    "tms_10x_tissuewise_knn.h5ad"
)
MUKIN_SAMPLE_LIST_PATH = Path(
    "/scratch/chenxi.sun/thesis_project/TIDYI/TIDYI_pattern/"
    "mukin_sample_stas.csv"
)
TMS_SAMPLE_LIST_PATH = Path(
    "/scratch/chenxi.sun/thesis_project/TIDYI/TIDYI_pattern/"
    "tms_sample_stas.csv"
)

mukin_samples = pd.read_csv(MUKIN_SAMPLE_LIST_PATH)["Sample"].tolist()
tms_samples = pd.read_csv(TMS_SAMPLE_LIST_PATH)["sample"].tolist()

aer_mean_umi, aer_mean_umi_figure = plot_sample_mean_umi(
    adata_path=AER_FINAL_PATH,
    sample_col="sample",
    output_path=(
        OUT["sample_mean_umi"] / "aer_sample_mean_umi_barplot.png"
    ),
    title="AER DISTILL: mean UMI per retained cell by sample",
    color="#4C78A8",
)
mukin_mean_umi, mukin_mean_umi_figure = plot_sample_mean_umi(
    adata_path=MUKIN_KNN_PATH,
    sample_col="Sample",
    include_samples=mukin_samples,
    output_path=(
        OUT["sample_mean_umi"] / "mukin_sample_mean_umi_barplot.png"
    ),
    title="Mukin: mean UMI per retained cell by sample",
    color="#59A14F",
)
tms_mean_umi, tms_mean_umi_figure = plot_sample_mean_umi(
    adata_path=TMS_KNN_PATH,
    sample_col="sample",
    include_samples=tms_samples,
    output_path=(
        OUT["sample_mean_umi"] / "tms_sample_mean_umi_barplot.png"
    ),
    title="TMS: mean UMI per retained cell by sample",
    color="#F28E2B",
)

sample_mean_umi_summary = pd.concat(
    [
        aer_mean_umi.assign(dataset="AER"),
        mukin_mean_umi.assign(dataset="Mukin"),
        tms_mean_umi.assign(dataset="TMS"),
    ],
    ignore_index=True,
)[["dataset", "sample", "n_cells", "mean_umi"]]
display(sample_mean_umi_summary)
plt.show()